In [3]:
import os
api_key="vp_3211809e.acae6882f57eab5647963fb1fac1b8ed235642111c10b956ba2209385fc93b58"
model_name="gemini-3.1-flash-lite-preview"

In [5]:
# load PDFs

from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader


documents_dir = Path("../../data/sec10k")
print(documents_dir)
if not documents_dir.is_dir():
    raise FileNotFoundError(f"Documents folder not found: {documents_dir.resolve()}")

pdf_paths = sorted(
    (path for path in documents_dir.rglob("*") if path.is_file() and path.suffix.lower() == ".pdf"),
    key=lambda path: path.as_posix().lower(),
)
if not pdf_paths:
    raise FileNotFoundError(f"No PDF files found in {documents_dir.resolve()}")

documents = []
for pdf_path in pdf_paths:
    loaded_documents = PyPDFLoader(str(pdf_path), mode="single").load()
    for document in loaded_documents:
        document.metadata["source"] = str(pdf_path)
        document.metadata["file_name"] = pdf_path.name
    documents.extend(loaded_documents)

print(f"Loaded {len(documents)} PDFs as Document objects")

C:\Users\artyv\AppData\Local\Temp\ipykernel_24568\187462895.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


..\..\data\sec10k
Loaded 3 PDFs as Document objects


In [6]:
# text chunking

from langchain_text_splitters import RecursiveCharacterTextSplitter
import json

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
)

chunks = text_splitter.split_documents(documents)
chunk_dicts = [chunk.model_dump() for chunk in chunks] # Convert the list of Document objects into a list of dictionaries
print(f"Split into {len(chunks)} chunks")

with open("rp01_chunks.json", "w", encoding="utf-8") as file:
    json.dump(
        {"chunks": chunk_dicts},
        file,
        ensure_ascii=False,
        indent=2,
    )

print(f"Saved {len(chunks)} chunks to rp01_chunks.json")

Split into 118 chunks
Saved 118 chunks to rp01_chunks.json


In [7]:
# create embeddings using openai text-embedding-3-small model

# from langchain_huggingface import HuggingFaceEmbeddings
from tqdm.auto import tqdm
from openai import OpenAI

# Initialize the client. It automatically picks up the OPENAI_API_KEY environment variable.
client = OpenAI(base_url="https://proxy.vectorinstitute.ai/v1",api_key=api_key)

vectors = []
for chunk in tqdm(chunks, desc="Embedding chunks", unit="chunk"):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=chunk.page_content,
    ) 
    embedding_vector = response.data[0].embedding
    vectors.append(embedding_vector)

with open("rp01_embeddings.json", "w", encoding="utf-8") as file:
    json.dump(
        [
            {
                "metadata": chunk.metadata,
                "embedding": vector,
            }
            for chunk, vector in zip(chunks, vectors)
        ],
        file,
        ensure_ascii=False,
        indent=2,
    )

print(f"Saved {len(vectors)} embeddings to rp01_embeddings.json") 

Embedding chunks:   0%|          | 0/118 [00:00<?, ?chunk/s]

Embedding chunks: 100%|██████████| 118/118 [00:38<00:00,  3.03chunk/s]


Saved 118 embeddings to rp01_embeddings.json


In [ ]:
# test gemini-3.1-flash-lite-preview model 

import os

from dotenv import load_dotenv
from openai import OpenAI


# Loads your environment variables from the .env file
load_dotenv()
client = OpenAI(base_url="https://proxy.vectorinstitute.ai/v1", api_key=api_key)

stream = client.chat.completions.create(
    model="gemini-3.1-flash-lite-preview",
    messages=[{"role": "user", "content": "Hi, how are you?"}],
    stream=True,
)

for chunk in stream:
    if chunk.choices:
        print(chunk.choices[0].delta.content or "", end="\n", flush=True)


print("<--DEFAULT MODELS-->")
print(f"GENERATOR_MODEL{os.getenv('GENERATOR_MODEL')}")
print(f"AGENT_MODEL{os.getenv('AGENT_MODEL')}")
print(f"USER_SIMULATOR_MODEL{os.getenv('USER_SIMULATOR_MODEL')}")


I'm doing great, thank you for asking! How are you doing today?
 Is there anything I can help you with?

<--DEFAULT MODELS-->
GENERATOR_MODELNone
AGENT_MODELNone
USER_SIMULATOR_MODELNone
